
Привет, меня зовут Люман Аблаев. Сегодня я проверю твой проект.
<br> Дальнейшее общение будет происходить на "ты" если это не вызывает никаких проблем.
<br> Желательно реагировать на красные комментарии ('исправил', 'не понятно как исправить ошибку', ...)
<br> Пожалуйста, не удаляй комментарии ревьюера, так как они повышают качество повторного ревью.

Комментарии будут в <font color='green'>зеленой</font>, <font color='blue'>синей</font> или <font color='red'>красной</font> рамках:


<div class="alert alert-block alert-success">
<b>Успех:</b> Если все сделано отлично
</div>

<div class="alert alert-block alert-info">
<b>Совет: </b> Если можно немного улучшить
</div>

<div class="alert alert-block alert-danger">
<b>Ошибка:</b> Если требуются исправления. Работа не может быть принята с красными комментариями.
</div>

-------------------

Будет очень хорошо, если ты будешь помечать свои действия следующим образом:
<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> ...
</div>

<div class="alert alert-block alert-warning">
<b>Изменения:</b> Были внесены следующие изменения ...
</div>







<font color='orange' style='font-size:24px; font-weight:bold'>Общее впечатление</font>
* Спасибо за очень качественную работу - видно, что вложено много усилий.
- Я оставил некоторые советы, надеюсь они будут полезными и интересными
- Есть некоторые недочеты, которые нужно поправить, но у тебя это не должно занять много времени)
- Жду обновленную работу






# Анализ лояльности пользователей Яндекс Афиши




 <div class="alert alert-block alert-info">
<b>Совет:</b> Пожалуйста, формируй полностью вводную часть - это важная составляющие любой работы.  Нужны цели и задачи, описание данных, содержание проекта.
</div>

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

- **Хост** — `rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net`
- **База данных** — `data-analyst-afisha`
- **Порт** — `6432`
- **Аутентификация** — `Database Native`
- **Пользователь** — `praktikum_student`
- **Пароль** — `Sdf4$2;d-d30pp`

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


**a) Импортируем необходимые библиотеки для подключения к базе данных:**

In [1]:
import pandas as pd

In [2]:
!pip install sqlalchemy

You should consider upgrading via the '/Users/alexandr/Documents/project2itog/venv/bin/python3 -m pip install --upgrade pip' command.


In [3]:
!pip install psycopg2-binary

You should consider upgrading via the '/Users/alexandr/Documents/project2itog/venv/bin/python3 -m pip install --upgrade pip' command.


In [4]:
from sqlalchemy import create_engine 

In [5]:
!pip install python-dotenv 

You should consider upgrading via the '/Users/alexandr/Documents/project2itog/venv/bin/python3 -m pip install --upgrade pip' command.


In [6]:
from dotenv import load_dotenv
import os

# Загрузка .env
load_dotenv()

# Конфиг
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'db': os.getenv('DB_NAME')
}

In [7]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

In [8]:
engine = create_engine(connection_string)


<div class="alert alert-block alert-info">
<b>Совет:</b> Советую собирать все импорты в верхней части ноутбука! 
Если у того, кто будет запускать твой ноутбук будут отсутствовать некоторые библиотеки, то он это увидит сразу, а не в процессе!
</div>


**б) Вводим параметры подключения к БД и подключаемся:**

**в) Запрашиваем необходимые для дальнейших задач данные и выводим их**

In [9]:
# SQL-запрос
query = """
SELECT 
   user_id,
    device_type_canonical,   
    order_id,
    created_dt_msk AS order_dt,
    created_ts_msk AS order_ts,
    currency_code,
    revenue,
    tickets_count,
    created_dt_msk::date - LAG(p.created_dt_msk::date) 
        OVER(PARTITION BY user_id ORDER BY p.created_dt_msk)
        AS days_since_prev,
    event_id,
    service_name,
    e.event_name_code AS event_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases AS p
JOIN afisha.events AS e USING (event_id)
JOIN afisha.city AS c USING(city_id)
JOIN afisha.regions AS r USING(region_id)
WHERE
    device_type_canonical IN ('mobile','desktop') 
    AND e.event_type_main != 'фильм'
ORDER BY user_id;
"""

df = pd.read_sql_query(query, con=engine)

# Просмотр первых 10 строк
df.head(10)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,service_name,event_name,event_type_main,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,Край билетов,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,Мой билет,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,За билетом!,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,Лови билет!,2f638715-8844-466c-b43f-378a627c419f,другое,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,Билеты без проблем,10d805d3-9809-4d8a-834e-225b7d03f95d,стендап,Озернинский край,Родниковецк
5,000898990054619,mobile,2613713,2024-10-23,2024-10-23 15:12:00,rub,902.74,3,19.0,500862,Облачко,9cc55c15-4375-4129-9979-3129688ba1b4,концерты,Лугоградская область,Кристалевск
6,00096d1f542ab2b,desktop,6636941,2024-08-15,2024-08-15 16:48:48,rub,917.83,4,NaN,201953,Край билетов,2f98d69f-4e60-4ffc-8f16-e539383526b1,театр,Каменевский регион,Глиногорск
7,000a55a418c128c,mobile,4657981,2024-09-29,2024-09-29 19:39:12,rub,47.78,1,NaN,265857,Лучшие билеты,0d876e01-851e-458b-ba61-753e0e0c4063,театр,Поленовский край,Дальнозолотск
8,000a55a418c128c,mobile,4657952,2024-10-15,2024-10-15 10:29:04,rub,74.84,2,16.0,271579,Лучшие билеты,ddc795f8-7ef8-4eb0-b299-cb3e6ee24ba1,театр,Поленовский край,Дальнозолотск
9,000cf0659a9f40f,mobile,6818017,2024-06-20,2024-06-20 10:35:26,rub,1421.91,4,NaN,516728,Лови билет!,11be386f-7cb7-4aa1-a8e4-ba73a29c1af2,концерты,Широковская область,Радужнополье


<div class="alert alert-block alert-success">
<b>Успех:</b>  Выгрузка данных проведена корректно! Была выполнена необходимая фильтрация данных, выгружены только необходимые для анализа данные. Отлично, что сразу подсчитываешь время между заказами для каждого пользователя.
</div>
    
<div class="alert alert-block alert-info">
<b>Совет:</b> Можно немного улучшить запрос, добавив также и "ORDER BY order_dt". Так внутри каждого пользователя заказы будут идти строго по дате.
</div>



---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [10]:
# Общая информация о данных
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  service_name           290611 non-null  object        
 11  event_name             290611 non-null  object        
 12  event_type_main        290611 non-null  obje

In [11]:
# Проверка на пропущенные значения
df.isnull().sum()

user_id                      0
device_type_canonical        0
order_id                     0
order_dt                     0
order_ts                     0
currency_code                0
revenue                      0
tickets_count                0
days_since_prev          21933
event_id                     0
service_name                 0
event_name                   0
event_type_main              0
region_name                  0
city_name                    0
dtype: int64

***Промежуточный вывод***

*На этом этапе мы имеем рабочую таблицу с данными о заказах пользователей, которая включает:*

- 15 столбцов с различными типами данных;
- Пропущенные значения только в столбце days_since_prev;
- Необходимость проведения предобработки для нормализации данных, обработки пропусков, преобразования типов данных и проверки на выбросы.

*Таким образом, необходимо выполнить следующие шаги:*

- Обработать пропуски в days_since_prev;
- Проверить и преобразовать типы данных, особенно для числовых столбцов, чтобы уменьшить размерность;
- Выполнить очистку данных от дубликатов;
- Провести обработку категориальных признаков для подготовки к анализу.


<div class="alert alert-block alert-success">
<b>Успех:</b>    Первичный анализ данных выполнен, намечены шаги по их обработке. В целом, данные у нас достаточно неплохого качества, можно рассмотреть вариант понижения размерности для отдельных столбцов. В реальной практике это иногда бывает очень полезным) 
</div>


---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [12]:
# Загрузка данных о курсе тенге к рублю
tenge_df = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [13]:
tenge_df

,data,nominal,curs,cdx
0,2024-01-10,100,19.9391,kzt
1,2024-01-11,100,19.7255,kzt
2,2024-01-12,100,19.5839,kzt
3,2024-01-13,100,19.4501,kzt
4,2024-01-14,100,19.4501,kzt
...,...,...,...,...
352,2024-12-27,100,19.2705,kzt
353,2024-12-28,100,19.5105,kzt
354,2024-12-29,100,19.4860,kzt
355,2024-12-30,100,19.4860,kzt


In [14]:
# Преобразуем 'date' в pandas datetime
tenge_df['data'] = pd.to_datetime(tenge_df['data'])

# Выполняем объединение данных по дате (уже после преобразования обоих столбцов в datetime)
df_merged = pd.merge(df, tenge_df[['data', 'curs']], left_on='order_dt', right_on='data', how='left')

# Преобразуем выручку в рубли, если валюта KZT
df_merged['revenue_rub'] = df_merged.apply(
    lambda row: row['revenue'] * row['curs'] / 100 if row['currency_code'] == 'kzt' else row['revenue'],
    axis=1
)

# Проверка столбца revenue_rub
df_merged['revenue_rub'].head(10)

0    1521.94
1     289.45
2    1258.57
3       8.49
4    1390.41
5     902.74
6     917.83
7      47.78
8      74.84
9    1421.91
Name: revenue_rub, dtype: float64

<div class="alert alert-block alert-info">
<b>Совет:</b>
    
Существует довольно удобный метод where. Мы можем применить его к столбцу и указать условие, которое будем проверять, а также альтернативный вариант значения. Если условие выполняется, то берется значение из столбца, если нет - альтернативное значение. Тогда расчет выручки в рублях будет выглядеть следующим образом:
    
```
df['revenue_rub'] = df['revenue'].where(df['currency_code'] == 'rub', df['revenue'] * df['curs'] / 100)
```
</div>

In [15]:
df_merged.head(10)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,service_name,event_name,event_type_main,region_name,city_name,data,curs,revenue_rub
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,Край билетов,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Каменевский регион,Глиногорск,2024-08-20,18.6972,1521.94
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,Мой билет,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Каменевский регион,Глиногорск,2024-07-23,18.3419,289.45
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,За билетом!,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,Каменевский регион,Глиногорск,2024-10-06,19.6475,1258.57
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,Лови билет!,2f638715-8844-466c-b43f-378a627c419f,другое,Североярская область,Озёрск,2024-07-13,18.5010,8.49
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,Билеты без проблем,10d805d3-9809-4d8a-834e-225b7d03f95d,стендап,Озернинский край,Родниковецк,2024-10-04,19.6648,1390.41
5,000898990054619,mobile,2613713,2024-10-23,2024-10-23 15:12:00,rub,902.74,3,19.0,500862,Облачко,9cc55c15-4375-4129-9979-3129688ba1b4,концерты,Лугоградская область,Кристалевск,2024-10-23,20.0531,902.74
6,00096d1f542ab2b,desktop,6636941,2024-08-15,2024-08-15 16:48:48,rub,917.83,4,NaN,201953,Край билетов,2f98d69f-4e60-4ffc-8f16-e539383526b1,театр,Каменевский регион,Глиногорск,2024-08-15,18.7730,917.83
7,000a55a418c128c,mobile,4657981,2024-09-29,2024-09-29 19:39:12,rub,47.78,1,NaN,265857,Лучшие билеты,0d876e01-851e-458b-ba61-753e0e0c4063,театр,Поленовский край,Дальнозолотск,2024-09-29,19.3741,47.78
8,000a55a418c128c,mobile,4657952,2024-10-15,2024-10-15 10:29:04,rub,74.84,2,16.0,271579,Лучшие билеты,ddc795f8-7ef8-4eb0-b299-cb3e6ee24ba1,театр,Поленовский край,Дальнозолотск,2024-10-15,19.7185,74.84
9,000cf0659a9f40f,mobile,6818017,2024-06-20,2024-06-20 10:35:26,rub,1421.91,4,NaN,516728,Лови билет!,11be386f-7cb7-4aa1-a8e4-ba73a29c1af2,концерты,Широковская область,Радужнополье,2024-06-20,18.0419,1421.91


In [16]:
# Удаляем столбцы 'data' и 'curs' из DataFrame
df_merged = df_merged.drop(columns=['data', 'curs'])

# Проверим, что столбцы удалены
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 290611 entries, 0 to 290610
Data columns (total 16 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  service_name           290611 non-null  object        
 11  event_name             290611 non-null  object        
 12  event_type_main        290611 non-null  obje

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [17]:
# Проверка на пропущенные значения в данных
missing_data = df_merged.isnull().sum()

# Печать информации о пропущенных значениях
missing_data

user_id                      0
device_type_canonical        0
order_id                     0
order_dt                     0
order_ts                     0
currency_code                0
revenue                      0
tickets_count                0
days_since_prev          21933
event_id                     0
service_name                 0
event_name                   0
event_type_main              0
region_name                  0
city_name                    0
revenue_rub                  0
dtype: int64

**!!!Делаем вывод, что выгрузка является правильной!!!**

In [18]:
# Оптимизация числовых столбцов
df_merged['order_id'] = df_merged['order_id'].astype('int32')  # Понижаем размерность до int32
df_merged['event_id'] = df_merged['event_id'].astype('int32')  # Понижаем размерность до int32
df_merged['tickets_count'] = df_merged['tickets_count'].astype('int16')  # Понижаем размерность до int16, если значения не превышают 65535
df_merged['revenue'] = df_merged['revenue'].astype('float32')  # Понижаем размерность до float32

***Объяснение изменений:***
- order_id и event_id преобразуем в int32, так как идентификаторы заказов и мероприятий не выходят за пределы диапазона int32.
- tickets_count преобразуем в int16 (если предполагается, что количество билетов не превысит 65535).
- revenue преобразуем в float32, что уменьшит потребление памяти без потери точности для большинства аналитических задач.

In [19]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 290611 entries, 0 to 290610
Data columns (total 16 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  object        
 1   device_type_canonical  290611 non-null  object        
 2   order_id               290611 non-null  int32         
 3   order_dt               290611 non-null  datetime64[ns]
 4   order_ts               290611 non-null  datetime64[ns]
 5   currency_code          290611 non-null  object        
 6   revenue                290611 non-null  float32       
 7   tickets_count          290611 non-null  int16         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int32         
 10  service_name           290611 non-null  object        
 11  event_name             290611 non-null  object        
 12  event_type_main        290611 non-null  obje

In [20]:
# Проверка на отрицательные значения в ключевых столбцах
df_merged[df_merged['revenue_rub'] < 0]

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,service_name,event_name,event_type_main,region_name,city_name,revenue_rub
252,00eb3dc9baa1543,mobile,1594653,2024-06-29,2024-06-29 15:01:43,rub,-2.37,3,NaN,538650,Билеты без проблем,ffe03bc6-0e0c-480d-b037-6a4b55540ab5,другое,Берёзовская область,Златопольск,-2.37
4539,02ea4583333f064,mobile,2360920,2024-09-03,2024-09-03 18:12:58,rub,-0.23,3,0.0,559772,Билеты без проблем,592856bb-09a5-4d32-9534-0e02c6056e44,другое,Широковская область,Лесореченск,-0.23
4544,02ea4583333f064,mobile,2361094,2024-09-04,2024-09-04 09:34:53,rub,-0.15,2,0.0,559772,Билеты без проблем,592856bb-09a5-4d32-9534-0e02c6056e44,другое,Широковская область,Лесореченск,-0.15
8135,043f669c9f734b1,mobile,166780,2024-09-27,2024-09-27 10:00:09,rub,-1.86,3,0.0,567183,Лучшие билеты,9f571dad-b18a-4095-ac76-9db60d8dd97a,другое,Золотоключевской край,Луговинец,-1.86
8136,043f669c9f734b1,mobile,166809,2024-09-27,2024-09-27 10:56:35,rub,-0.62,1,0.0,567183,Лучшие билеты,9f571dad-b18a-4095-ac76-9db60d8dd97a,другое,Золотоключевской край,Луговинец,-0.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288837,fe237d2cfd6e503,mobile,3700575,2024-10-12,2024-10-12 08:11:33,rub,-5.70,1,0.0,247058,Тебе билет!,a3214473-934e-44ad-a8da-82915f51583f,концерты,Речицкая область,Радужанов,-5.70
288910,fe237d2cfd6e503,desktop,3523617,2024-10-15,2024-10-15 20:13:13,rub,-0.96,2,0.0,243963,Лучшие билеты,80f5f95b-9a58-401d-b888-4261335ae290,другое,Ягодиновская область,Речинцево,-0.96
288911,fe237d2cfd6e503,desktop,3523646,2024-10-15,2024-10-15 20:32:18,rub,-1.43,3,0.0,243963,Лучшие билеты,80f5f95b-9a58-401d-b888-4261335ae290,другое,Ягодиновская область,Речинцево,-1.43
289058,fe237d2cfd6e503,mobile,5445853,2024-10-21,2024-10-21 20:22:29,rub,-0.61,1,0.0,243393,Лучшие билеты,1f30acba-8b62-41c3-aaea-a80bf58d0d26,другое,Ягодиновская область,Речинцево,-0.61


In [21]:
# Проверка на отрицательные значения в tickets_count
df_merged[df_merged['tickets_count'] < 0]

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,service_name,event_name,event_type_main,region_name,city_name,revenue_rub


***Обнаружены отрицательные значения, но их столь мало, что их необработка не повлияет на результат. Также, отрицательная выручка может сигнализировать об избегании больших потерь при продаже билетов после возврата, что тоже стоит принимать в рассчет.***


<div class="alert alert-block alert-success">
<b>Успех:</b>  всё по существу.

Можно только добавить пару деталей, чтобы закрыть вопрос с отрицательными значениями:

Сейчас ты написал, что отрицательная выручка может быть связана с разными причинами. Хорошо бы еще указать, сколько таких заказов в данных. Например: "Отрицательных значений оказалось всего Х штук (менее Y% от выборки) — это единичные случаи, поэтому на общий анализ они не повлияют, но иметь в виду их стоит".



</div>

In [22]:
# Проверка уникальных значений для категориальных столбцов
df_merged['device_type_canonical'].value_counts()

mobile     232490
desktop     58121
Name: device_type_canonical, dtype: int64

In [23]:
df_merged['currency_code'].value_counts()

rub    285542
kzt      5069
Name: currency_code, dtype: int64

In [24]:
df_merged['service_name'].value_counts()

Билеты без проблем        63519
Лови билет!               41124
Билеты в руки             40343
Мой билет                 34839
Облачко                   26642
Лучшие билеты             17774
Весь в билетах            16849
Прачечная                 10273
Край билетов               6207
Тебе билет!                5228
Яблоко                     5039
Дом культуры               4502
За билетом!                2865
Городской дом культуры     2733
Show_ticket                2200
Мир касс                   2167
Быстробилет                2003
Выступления.ру             1616
Восьмёрка                  1118
Crazy ticket!               790
Росбилет                    539
Шоу начинается!             499
Быстрый кассир              381
Радио ticket                376
Телебилет                   321
КарандашРУ                  133
Реестр                      125
Билет по телефону            85
Вперёд!                      80
Дырокол                      74
Кино билет                   67
Цвет и б

In [25]:
df_merged['event_type_main'].value_counts()

концерты    115276
театр        67321
другое       65867
спорт        21911
стендап      13393
выставки      4854
ёлки          1989
Name: event_type_main, dtype: int64

In [26]:
df_merged['region_name'].value_counts()

Каменевский регион          91058
Североярская область        44049
Широковская область         16457
Медовская область           13901
Озернинский край            10476
                            ...  
Лесноярский край               19
Крутоводский регион            18
Верхозёрский край              11
Сосноводолинская область       10
Теплоозёрский округ             7
Name: region_name, Length: 81, dtype: int64

***В целом, повторов и дубликатов(явных и неявных) не обнаружено***

***Посторим гистограммы распределения значений и диаграммы размаха для выручки и кол-ва билетов:***


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Гистограмма для 'revenue_rub'
plt.figure(figsize=(10, 6))
sns.histplot(df_merged['revenue_rub'], kde=True, bins=5)
plt.title('Распределение выручки (revenue_rub)')
plt.xlabel('Выручка (руб.)')
plt.ylabel('Частота')
plt.show()

# Гистограмма для 'tickets_count'
plt.figure(figsize=(10, 6))
sns.histplot(df_merged['tickets_count'], kde=True, bins=10)
plt.title('Распределение количества билетов (tickets_count)')
plt.xlabel('Количество билетов')
plt.ylabel('Частота')
plt.show()

# Диаграмма размаха для выручки
plt.figure(figsize=(10, 6))
sns.boxplot(x=df_merged['revenue_rub'])
plt.title('Диаграмма размаха для выручки')
plt.xlabel('Выручка (руб.)')
plt.show()

# Диаграмма размаха для количества билетов
plt.figure(figsize=(10, 6))
sns.boxplot(x=df_merged['tickets_count'])
plt.title('Диаграмма размаха для количества билетов')
plt.xlabel('Количество билетов')
plt.show()

In [ ]:
# Вычисление 99-го перцентиля для выручки
revenue_percentile_99 = df_merged['revenue_rub'].quantile(0.99)
revenue_percentile_99

In [ ]:
# Фильтрация данных по 99 перцентилю
df_filtered = df_merged[df_merged['revenue_rub'] <= revenue_percentile_99]

In [ ]:
# Сравниваем количество строк до и после фильтрации
print("Объем данных до фильтрации:", df_merged.shape[0])
print("Объем данных после фильтрации:", df_filtered.shape[0])

# Процент отфильтрованных данных
filtered_percentage = (df.shape[0] - df_filtered.shape[0]) / df.shape[0] * 100
print(f"Процент отфильтрованных данных: {filtered_percentage:.2f}%")

***Ответы на поставленные вопросы:***

1. Распределение численных данных и выбросы:
- Для данных о выручке (revenue_rub) и количестве билетов (tickets_count) были построены гистограммы распределения и диаграммы размаха.
- Обе переменные имеют явные выбросы, как видно на диаграммах размаха.
- Для выручки большинство заказов имеют низкую выручку, а несколько наблюдений имеют крайне высокие значения (выбросы).
- Для количества билетов наблюдаются несколько значений, которые сильно выбиваются из общего ряда (например, очень высокие значения количества билетов).
2. Отфильтрованные данные:
- Были отфильтрованы строки с выручкой выше 99-го перцентиля, что составляет 2628.42 рубля.
- После фильтрации данных объем уменьшился на 0.97% — это малый процент, что свидетельствует о небольшом количестве выбросов.
3. Выводы:
- Данные в целом корректны, и выбросы в столбцах revenue_rub и tickets_count были успешно обработаны, отфильтрованы.
- Объем данных остался в основном неизменным (менее 1% удаленных строк).
- Сформированы новые данные (отфильтрованные данные) для дальнейшего анализа.

*Данные готовы для дальнейшего анализа и моделирования.*


<div class="alert alert-block alert-success">
<b>Успех:</b>   Хвалю за то, что сразу проверяешь объем удаленных данных. Это помогает "не увлечься")

</div>



<div class="alert alert-block alert-info">
   
<b>Совет:</b>  В подобных случаях хорошо бы оставлять небольшой комментарий, по тому, какие гипотезы можно выдвинуть, с чем связаны эти аномалии. Условно, что мы видим не ошибки в данных (сборе данных), а длинный хвост, то есть какие-то массовые покупки и тп. Они нам не нужны в контексте задачи.
</div>


---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


***Сортируем данные как указанно в рекомендации к поставленной задаче:***

In [ ]:
# Сортируем данные по user_id и order_dt
df_sorted = df_filtered.sort_values(by=['user_id', 'order_dt'])
df_sorted

***Группировка данных по пользователям для создания профиля пользователя***

In [ ]:
# Группировка данных по пользователям для создания профиля
user_profile_unfil = df_sorted.groupby('user_id').agg(
    first_order_date=('order_dt', 'min'),
    last_order_date=('order_dt', 'max'),
    first_device_type=('device_type_canonical', 'first'),
    first_region=('region_name', 'first'),
    first_service_name=('service_name', 'first'),
    first_event_type=('event_type_main', 'first'),
    total_orders=('order_id', 'count'),
    avg_revenue=('revenue_rub', 'mean'),
    avg_tickets_count=('tickets_count', 'mean'),
    avg_days_between_orders=('days_since_prev', 'mean')
).reset_index()
user_profile_unfil

***Создаем два бинарных признака:***

- is_two — совершил ли пользователь 2 и более заказа;
- is_five — совершил ли пользователь 5 и более заказов.

In [ ]:
# Создание бинарных признаков
user_profile_unfil['is_two'] = (user_profile_unfil['total_orders'] >= 2).astype(int)
user_profile_unfil['is_five'] = (user_profile_unfil['total_orders'] >= 5).astype(int)

# Просмотр первых 10 строк профиля пользователей
user_profile_unfil.head(10)

***По результатам выполнения задачи получен "профиль пользователя", который может использоваться для построения модели прогнозирования возврата пользователей***


<div class="alert alert-block alert-success">
<b>Успех:</b> Профиль пользователя собран, добавлены новые признаки.

</div>

<div class="alert alert-block alert-info">
<b>Совет:</b>   Могу показать и такой вариант формирования профиля здесь:
    
    profiles = (df
            # В начале сортируем данные по дате совершения заказа, что найти первые признаки:
            .sort_values(by='order_ts')
            # Затем группируем по номеру пользователя и агрегируем данные:
            .groupby('user_id')
            .agg(
                # Находим первую и последнюю даты заказа:
                first_order_dt=('order_dt','min'),
                last_order_dt=('order_dt','max'),
                # Находим устройства, регион и название билетного партнера первого заказа:
                first_device=('device_type_canonical','first'),
                first_region_name=('region_name','first'),
                first_service_name=('service_name','first'),
                # Жанр первого посещённого мероприятия (event_type_main):
                first_event_type=('event_type_main','first'),
                # Подсчитваем количество заказов:
                total_orders=('order_id','nunique'),
                # Считаем статистику по заказам: средняя стоимость заказа, среднее количество билетов:
                avg_revenue_rub=('revenue_rub','mean'),
                avg_tickets_count=('tickets_count','mean'),
                # Считаем среднее количество дней между покупками:
                avg_days_since_prev=('days_since_prev','mean')
            )
            # Создаем два признака: совершил ли пользователь 2 / 5 и более заказов:
            .assign(
                is_two = lambda x: x['total_orders'] >= 2,
                is_five = lambda x: x['total_orders'] >= 5
            )
            # Можно альтернативным образом подсчитать среднее количество дней между заказами (если не будет в SQL):
            .assign(
                avg_days = lambda x: (x['last_order_dt'] - x['first_order_dt']).dt.days / (x['total_orders'] - 1)
            )
            .reset_index()
)

Почитать про assign более подробно можно [здесь](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
</div>


---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

***1.Рассчитываем общее количество уникальных пользователей***

In [ ]:
# Рассчитываем общее количество уникальных пользователей
total_users = user_profile_unfil['user_id'].nunique()
total_users

***2.Рассчитываем среднюю выручку с одного заказа***

In [ ]:
# Рассчитываем среднюю выручку с одного заказа
avg_revenue_per_order = user_profile_unfil['avg_revenue'].mean()
avg_revenue_per_order

***3.Рассчитывем долю пользователей, совершивших 2 и 5 более заказов***

In [ ]:
# Доля пользователей, совершивших 2 и более заказа
users_two_or_more_orders = (user_profile_unfil['is_two'] == 1).mean()

# Доля пользователей, совершивших 5 и более заказов
users_five_or_more_orders = (user_profile_unfil['is_five'] == 1).mean()

# Выводим результаты
print(f"Доля пользователей, совершивших 2 и более заказа: {users_two_or_more_orders:.2f}")
print(f"Доля пользователей, совершивших 5 и более заказов: {users_five_or_more_orders:.2f}")

***4. Проанализируем данные на аномалии и, при необходимости, проведем фильтраццию***

In [ ]:
user_profile_unfil.describe()

In [ ]:
# Диаграмма размаха для количества заказов
plt.figure(figsize=(10, 6))
sns.boxplot(x=user_profile_unfil['total_orders'])
plt.title('Диаграмма размаха для количества заказов')
plt.xlabel('Количество заказов')
plt.show()

In [ ]:
# Диаграмма размаха для количества билетов
plt.figure(figsize=(10, 6))
sns.boxplot(x=user_profile_unfil['avg_tickets_count'])
plt.title('Диаграмма размаха для количества билетов')
plt.xlabel('Количество билетов')
plt.show()

In [ ]:
# Отфильтровываем пользователей, которые совершили минимум два заказа
filtered_df = user_profile_unfil[user_profile_unfil['is_two'] == 1]
filtered_df

In [ ]:
# Строим диаграмму размаха для количества билетов (только для пользователей с минимум двумя заказами)
plt.figure(figsize=(10, 6))
sns.boxplot(x=filtered_df['avg_days_between_orders'])
plt.title('Диаграмма размаха для количества дней между покупками')
plt.xlabel('Количество дней между покупками')
plt.show()

***Из анализа 'total_orders' мы можем наблюдать явные выбросы (max:10181, for example). Поэтому мною принято решение о фильтрации данного столбца по 95 процентилю:***

<div class="alert alert-block alert-info">
    
<b>Совет:</b> Было бы лучше, если бы мы  не просто фиксировали аномалии, но и пытались найти им объяснение. 
</div>



In [ ]:
# Вычисление 99-го перцентиля для total_orders
total_orders_percentile_99 = user_profile_unfil['total_orders'].quantile(0.95)

# Фильтрация данных по 99 перцентилю
user_profile = user_profile_unfil[user_profile_unfil['total_orders'] <= total_orders_percentile_99]

# Проверим изменения в объеме данных
print("Объем данных до фильтрации:", user_profile_unfil.shape[0])
print("Объем данных после фильтрации:", user_profile.shape[0])

# Процент отфильтрованных данных
filtered_percentage = (user_profile_unfil.shape[0] - user_profile.shape[0]) / user_profile_unfil.shape[0] * 100
print(f"Процент отфильтрованных данных: {filtered_percentage:.2f}%")


<div class="alert alert-block alert-danger">
<b>Ошибка:</b> Вывести статистические показатели по обновлённому датасет  -  это правильное решение, но какие выводы мы можем сделать?  Было бы не лишним, как минимум, написать, насколько изменилась средняя выручка после удаления выбросов. 
</div>


In [ ]:
user_profile.describe()

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> После фильтрации данных, можно отметить, что средее количество билетов теперь равно 4, вместо 13 до преобразования. Средняя выручка практически не изменилась (544 руб. до и 546 руб. после). То есть, можно сделать вывод, что качесвто данных стало выше и позволит нам провести более точный дальнейший исследоватльский анализ данных
</div>


<div class="alert alert-block alert-success">
    
<b>Успех[2]:</b> 👍 Отличное дополнение
</div>


---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.

In [ ]:
user_profile



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
# Сгруппируем пользователей по каждому признаку
grouped_by_event_type = user_profile.groupby('first_event_type')['user_id'].nunique()
grouped_by_event_type

In [ ]:
segment_by_event_type = grouped_by_event_type / total_users * 100
segment_by_event_type

In [ ]:
grouped_by_device = user_profile.groupby('first_device_type')['user_id'].nunique()
grouped_by_device

In [ ]:
segment_by_device = grouped_by_device / total_users * 100
segment_by_device

In [ ]:
grouped_by_region = user_profile.groupby('first_region')['user_id'].nunique()
grouped_by_region

In [ ]:
segment_by_region = grouped_by_region / total_users * 100
segment_by_region

In [ ]:
grouped_by_service_name = user_profile.groupby('first_service_name')['user_id'].nunique()
grouped_by_service_name

In [ ]:
segment_by_service_name = grouped_by_service_name / total_users * 100
segment_by_service_name

***Вот анализ распределения пользователей по различным признакам:***

1. **По типу первого мероприятия**:

   * Наибольшее количество пользователей выбрало "концерты" (42%), затем идут "другое" (23%), "театр" (18%), "стендап" (5%) и другие типы мероприятий.
   * Наименьший процент пользователей участвует в мероприятиях типа "ёлки" (0.4%).

2. **По типу устройства**:

   * Большинство пользователей совершили первый заказ с мобильных устройств (78.7%), в то время как с десктопов — всего 16.3%.

3. **По региону проведения первого мероприятия**:

   * Большинство пользователей из Каменевского региона (19.5%), а также Широковская область (5.4%).
   * Некоторые регионы имеют очень малое количество пользователей, например, "Яснопольский округ" (0.005%).

4. **По билетному оператору, продавшему билеты на первый заказ**:

   * Наибольшее количество пользователей обрабатывает оператор "Билеты без проблем" (22.7%), а также "Мой билет" (13%) и "Лови билет!" (12.4%).
   * Некоторые операторы, такие как "Зе Бест!" или "Тех билет", обслуживают лишь очень малую долю пользователей (менее 0.1%).

***Равномерно ли распределены пользователи по сегментам?***

Нет, распределение пользователей по сегментам не является равномерным. Существуют сегменты с выраженными «точками входа», например, типы мероприятий с большой долей пользователей (концерты, другие), устройства (мобильные устройства), и операторы (Билеты без проблем, Мой билет, Лови билет!). Эти сегменты имеют значительно больше пользователей по сравнению с остальными.

***Вывод:***
Распределение пользователей по сегментам (особенно по типу устройства и типу мероприятия) не является равномерным.



<div class="alert alert-block alert-success">
<b>Успех:</b> Задание выполнено корректно, с наблюдениями согласен
    
---
    Можно создать пользовательскую функцию, чтобы не прописывать практически один и тот же код несколько раз
    
      def segment_summary(df, column):
          seg = (df.groupby(column).agg(users_count=('user_id', 'nunique')) .reset_index().sort_values('users_count', ascending=False))
          seg['users_share'] = (seg['users_count'] / seg['users_count'].sum() *100)
          seg['users_share'] = seg['users_share'].round(2)
          return seg

</div>
<div class="alert alert-block alert-info">
<b>Совет:</b>  Неплохо было бы добавить графики - это сделало бы раздел интереснее и красочнее
```
</div>



---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
# Группируем пользователей по признакам и рассчитываем долю вернувшихся пользователей
# Для каждого сегмента вычисляем долю пользователей, совершивших 2 и более заказа
grouped_by_event_type_return_rate = user_profile.groupby('first_event_type').apply(lambda x: (x['is_two'] == 1).mean())
grouped_by_device_return_rate = user_profile.groupby('first_device_type').apply(lambda x: (x['is_two'] == 1).mean())
grouped_by_region_return_rate = user_profile.groupby('first_region').apply(lambda x: (x['is_two'] == 1).mean())
grouped_by_service_name_return_rate = user_profile.groupby('first_service_name').apply(lambda x: (x['is_two'] == 1).mean())

# Выводим результаты
print("Доля пользователей, совершивших 2 и более заказа по типам мероприятий:")
print(grouped_by_event_type_return_rate)

print("\nДоля пользователей, совершивших 2 и более заказа по типам устройств:")
print(grouped_by_device_return_rate)

print("\nДоля пользователей, совершивших 2 и более заказа по регионам:")
print(grouped_by_region_return_rate)

print("\nДоля пользователей, совершивших 2 и более заказа по билетным операторам:")
print(grouped_by_service_name_return_rate)

In [ ]:
top_10_region = grouped_by_region_return_rate.sort_values(ascending=False).head(10)
top_10_region

In [ ]:
top_10_service_name = grouped_by_service_name_return_rate.sort_values(ascending=False).head(10)
top_10_service_name

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Тема для графиков
sns.set(style="whitegrid")

# Столбчатая диаграмма для типов мероприятий с разными цветами
plt.figure(figsize=(10, 6))
grouped_by_event_type_return_rate.plot(kind='bar', color=sns.color_palette("Set2", len(grouped_by_event_type_return_rate)), alpha=0.8)
plt.title('Доля пользователей, совершивших 2 и более заказа по типам мероприятий', fontsize=14)
plt.xlabel('Тип мероприятия', fontsize=12)
plt.ylabel('Доля пользователей', fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)  # Поворот меток по оси x для читаемости
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Добавление сетки для оси y
plt.tight_layout()  # Плотное размещение
plt.show()

# Столбчатая диаграмма для типов устройств с разными цветами
plt.figure(figsize=(10, 6))
grouped_by_device_return_rate.plot(kind='bar', color=sns.color_palette("Paired", len(grouped_by_device_return_rate)), alpha=0.8)
plt.title('Доля пользователей, совершивших 2 и более заказа по типам устройств', fontsize=14)
plt.xlabel('Тип устройства', fontsize=12)
plt.ylabel('Доля пользователей', fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Столбчатая диаграмма для регионов с разными цветами
plt.figure(figsize=(10, 6))
top_10_region.plot(kind='bar', color=sns.color_palette("viridis", len(top_10_region)), alpha=0.8)
plt.title('Доля пользователей, совершивших 2 и более заказа по регионам', fontsize=14)
plt.xlabel('Регион', fontsize=12)
plt.ylabel('Доля пользователей', fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Столбчатая диаграмма для билетных операторов с разными цветами
plt.figure(figsize=(10, 6))
top_10_service_name.plot(kind='bar', color=sns.color_palette("coolwarm", len(top_10_service_name)), alpha=0.8)
plt.title('Доля пользователей, совершивших 2 и более заказа по билетным операторам', fontsize=14)
plt.xlabel('Билетный оператор', fontsize=12)
plt.ylabel('Доля пользователей', fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

***Ответ на вопросы:***
1. Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
Из анализа доли пользователей, совершивших два и более заказа, видно, что:


По типам мероприятий наибольшая доля возврата у пользователей можно отметить, что выбор мероприятия в пределах погрешности не влияет на возрат пользователя.


По типам устройств пользователи мобильных устройств (59.1%) возвращаются чуть реже, чем пользователи с десктопов (62.2%). 


По регионам наибольшую долю возврата показывают такие регионы, как "Верхозёрский край" (100%) и "Озернопольская область" (88.9%). Скорее всего это связанно с минимальной выборкой.


По билетным операторам самые высокие доли возврата показали "Зе Бест!" (100%), "Быстрый кассир" (83.9%) и "Билет по телефону" (83.3%). Скорее всего это связанно с минимальной выборкой.


2. Наблюдаются ли успешные «точки входа»?
Да, такие сегменты наблюдаются:


По регионам можно выделить топ-10 представленный на графике.


Аналогичная ситуация и с агрегаторами билетов.

3. Интерпретация результатов:


Небольшие сегменты (например, "Зе Бест!" или "Верхозёрский край") могут показывать очень высокие показатели из-за ограниченности данных (меньше пользователей), что делает эти данные нестабильными и возможно не отражающими общей картины.


Более крупные сегменты, такие как "Билеты без проблем" или "Мобильные устройства", имеют более репрезентативные данные, что позволяет делать выводы о том, что пользователи этих сегментов чаще возвращаются на платформу.


4. Выводы:


Пользователи, совершившие покупки через такие операторы, как "Зе Бест!", "Быстрый кассир" и "Реестр", а также в регионах с высокими показателями возврата, чаще совершают повторные заказы.


Известно, что мобильные пользователи имеют немного меньшую склонность к возврату, чем десктопные.


Важно учитывать, что данные для малых сегментов могут быть нестабильными, поэтому они могут требовать более тщательной интерпретации или исключения для более надежных выводов.


На основе этих выводов можно направить усилия на улучшение опыта пользователей в меньших сегментах или в сегментах с меньшим возвратом для повышения общей лояльности.


<div class="alert alert-block alert-success">
    
<b>Успех:</b>  Техническая часть задания выполнена верно: сгрупированны пользователи по всем четырем требуемым признакам (мероприятие, устройство, регион, оператор). Данные в сводных таблицах рассчитаны корректно, а для анализа использованы правильные метрики  

Итоговый вывод соответствует цифрам.  Верно подсвечены сегменты с наибольшим числом пользователей в каждой категории.
</div>



---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

In [ ]:
# Фильтруем пользователей по типу мероприятия
sport_users = user_profile[user_profile['first_event_type'] == 'спорт']
concert_users = user_profile[user_profile['first_event_type'] == 'концерты']

# Рассчитываем долю пользователей с 2 и более заказами (is_two == 1)
sport_return_rate = (sport_users['is_two'] == 1).mean()
concert_return_rate = (concert_users['is_two'] == 1).mean()

# Выводим результаты
print(f"Доля пользователей, совершивших 2 и более заказа (спортивные мероприятия): {sport_return_rate:.2f}")
print(f"Доля пользователей, совершивших 2 и более заказа (концерты): {concert_return_rate:.2f}")

***Ответ:***
- С полной увереннностью может опровергнуть данную гипотезу исходя из выводов предыдущих задач и рассчета в этой

<div class="alert alert-block alert-success">
<b>Успех:</b> Согласен


</div>


In [ ]:
# Группируем по регионам и считаем количество пользователей в каждом регионе
region_user_counts = user_profile.groupby('first_region')['user_id'].nunique()

# Фильтруем регионы, в которых больше всего пользователей (например, верхний 20% по числу пользователей)
top_regions = region_user_counts[region_user_counts > region_user_counts.quantile(0.8)]

# Проверяем долю пользователей с 2 и более заказами в этих регионах
top_region_users = user_profile[user_profile['first_region'].isin(top_regions.index)]
top_region_return_rate = (top_region_users['is_two'] == 1).mean()

# Регионы с меньшим количеством пользователей
bottom_regions = region_user_counts[region_user_counts <= region_user_counts.quantile(0.2)]
bottom_region_users = user_profile[user_profile['first_region'].isin(bottom_regions.index)]
bottom_region_return_rate = (bottom_region_users['is_two'] == 1).mean()

# Выводим результаты
print(f"Доля пользователей, совершивших 2 и более заказа в регионах с высоким числом пользователей: {top_region_return_rate:.2f}")
print(f"Доля пользователей, совершивших 2 и более заказа в регионах с низким числом пользователей: {bottom_region_return_rate:.2f}")

***Ответ:***
- Исходя из расчета выше, могу согласиться с данной гипотезей. В более активных регионах вероятность возврата пользователей выше на 13%

<div class="alert alert-block alert-success">
<b>Успех:</b> Согласен


</div>


---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---



In [ ]:
# Фильтруем пользователей по числу заказов
one_order_users = user_profile[user_profile['is_two'] == 0]
one_order_users.describe()

In [ ]:
returned_users = user_profile[user_profile['is_two'] == 1]
returned_users.describe()

In [ ]:
# Строим гистограмму для пользователей, совершивших один заказ
plt.figure(figsize=(10, 6))
sns.histplot(one_order_users['avg_revenue'], bins=30, kde=True, color='skyblue', label='1 заказ', alpha=0.6, stat="density")

# Строим гистограмму для вернувшихся пользователей (2 и более заказов)
sns.histplot(returned_users['avg_revenue'], bins=30, kde=True, color='salmon', label='2 и более заказов', alpha=0.6, stat="density")

# Добавляем подписи
plt.title('Сравнение распределения средней выручки с билета для пользователей с 1 и 2 и более заказами', fontsize=14)
plt.xlabel('Средняя выручка с билета (руб)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Тип пользователей')
plt.show()

***Ответ на вопрос 1: "В каких диапазонах средней выручки концентрируются пользователи из каждой группы?"***

Из гистограммы видно, что:

Пользователи, совершившие только один заказ:
Средняя выручка с билета для этой группы в основном сосредоточена в низких диапазонах (до 500 рублей), с пиками в интервале 0–200 рублей.
Часть пользователей также попадает в более высокие диапазоны, но эта часть значительно меньше.
Пользователи, совершившие два и более заказа:
Эти пользователи показывают более широкий диапазон средней выручки, с некоторыми значениями, превышающими 1000 рублей.
Основная концентрация находится в среднем диапазоне (200–500 рублей), но также видно больше пользователей с высокой выручкой (1000 и выше), что говорит о том, что возвращающиеся пользователи могут тратить больше на билеты.


***Ответ на вопрос 2: "Есть ли различия между группами?"***

Да, различия между группами можно заметить:

Пользователи, совершившие один заказ: как правило, имеют меньшую выручку, а их распределение более сосредоточено в низких диапазонах. Это может говорить о том, что эти пользователи покупали дешевые билеты или совершали одноразовые покупки.
Пользователи, совершившие два и более заказа: эта группа показывает более высокий разброс в выручке, что может указывать на более высокую вовлеченность, а также на покупки более дорогих билетов.

Это подтверждается более выраженными пиками в гистограмме для вернувшихся пользователей, которые включают более высокие значения выручки.

Вывод:
Пользователи, совершившие один заказ, склонны тратить меньше на билеты (концентрация в диапазоне до 500 рублей).
Пользователи с двумя и более заказами имеют более широкий диапазон выручки и склонны тратить больше.

Таким образом, можно сделать вывод, что высокая выручка с билета связана с вероятностью возврата пользователя на платформу.



<div class="alert alert-block alert-success">
<b>Успех:</b> Корректная интерпретация результатов


</div>



<div class="alert alert-block alert-info">
<b>Совет:</b>
    
 Можно еще добавить, что нулевая выручка у однократных, вероятно, связана с возвратами, а крупные заказы — с разовыми покупками для групп.

</div>




---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
# Фильтруем пользователей, совершивших от 2 до 4 заказов
two_to_four_orders = user_profile[(user_profile['is_two'] == 1) & (user_profile['is_five'] == 0)]

# Фильтруем пользователей, совершивших 5 и более заказов
five_or_more_orders = user_profile[(user_profile['is_two'] == 1) & (user_profile['is_five'] == 1)]

# Выводим количество пользователей в каждой группе, чтобы проверить
print(f'Пользователи с 2–4 заказами: {two_to_four_orders.shape[0]}')
print(f'Пользователи с 5 и более заказами: {five_or_more_orders.shape[0]}')

In [ ]:
two_to_four_orders.describe()

In [ ]:
five_or_more_orders.describe()

In [ ]:
# Строим гистограмму для пользователей, совершивших от 2 до 4 заказов
plt.figure(figsize=(10, 6))
sns.histplot(two_to_four_orders['avg_revenue'], bins=30, kde=True, color='skyblue', label='2–4 заказа', alpha=0.6, stat="density")

# Строим гистограмму для пользователей, совершивших 5 и более заказов
sns.histplot(five_or_more_orders['avg_revenue'], bins=30, kde=True, color='salmon', label='5 и более заказов', alpha=0.6, stat="density")

# Добавляем подписи
plt.title('Сравнение распределения средней выручки с билета для пользователей с 2–4 и 5 и более заказами', fontsize=14)
plt.xlabel('Средняя выручка с билета (руб)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Тип пользователей')
plt.show()

На основе диаграммы и описания видно, что распределение выручки с билета среди пользователей, совершивших 2–4 заказа, значительно отличается от распределения среди пользователей, совершивших 5 и более заказов.

Вот основные наблюдения:

* Пользователи, которые совершили **2–4 заказа**, имеют большую концентрацию выручки в диапазоне до 1000 рублей. Это означает, что их заказы, скорее всего, меньше по стоимости.
* Пользователи, совершившие **5 и более заказов**, распределены более равномерно, с выручкой в диапазоне до 2000 рублей. У них встречаются как более дешевые заказы, так и более дорогие.

**Есть ли различия между группами?**

Да, различия есть:

1. **Пользователи с 2–4 заказами** имеют тенденцию к меньшим суммам выручки с билета (основная масса пользователей имеет выручку до 500-600 рублей).
2. **Пользователи с 5 и более заказами** имеют более широкий диапазон выручки и более высокую среднюю выручку, что может свидетельствовать о большем объеме заказов или более дорогих событиях.

Таким образом, можно утверждать, что пользователи с большим числом заказов в среднем делают более дорогие заказы, что и приводит к большему значению выручки.




<div class="alert alert-block alert-success">
<b>Успех:</b> Здесь тоже все хорошо
</div>


<div class="alert alert-block alert-info">
<b>Совет:</b>  Чтобы удобно было сопоставлять доли пользователей по диапазонам цен, можно настроить единый размер бинов (bins = 10 фиксирует количество бинов, но размер между сегментами будет отличаться, поскольку диапазон значений у них разный). Для этого в bins можно передать границы для формирования бинов с шагом: bins = range(min_value, max_value+1, 50), максимальное и минимальное значения при этом определяем на всей выборке, а не отдельно для каждого сегмента.
    
---
    
Если хочется копнуть глубже 
    
- **Matplotlib: настройка параметра `bins` в гистограмме**  
  https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

- **Seaborn: документация `histplot` с примерами**  
  https://seaborn.pydata.org/generated/seaborn.histplot.html

- **Real Python: подробный гайд по гистограммам (англ.)**  
  https://realpython.com/python-histograms/
</div>	


---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

***Категорируем пользователей через функцию:***

In [ ]:
def categorize_tickets_count(row):
    if 1 <= row['avg_tickets_count'] < 2:
        return '1-2 билета'
    elif 2 <= row['avg_tickets_count'] < 3:
        return '2-3 билета'
    elif 3 <= row['avg_tickets_count'] < 5:
        return '3-5 билетов'
    else:
        return '5 и более билетов'

In [ ]:
# Применяем функцию к столбцу avg_tickets_count
user_profile['ticket_segment'] = user_profile.apply(categorize_tickets_count, axis=1)
user_profile

In [ ]:
ticket_segment_absolute = user_profile['ticket_segment'].value_counts()
ticket_segment_absolute

In [ ]:
ticket_segment_prop = user_profile['ticket_segment'].value_counts(normalize=True) * 100
ticket_segment_prop

In [ ]:
# Подсчитаем количество пользователей и долю повторных заказов в каждой группе
segment_stats = user_profile.groupby('ticket_segment').agg(
    total_users=('user_id', 'count'),
    repeat_users=('is_two', 'sum')  # Пользователи с 2 и более заказами
)

# Рассчитаем долю пользователей с 2 и более заказами
segment_stats['repeat_rate'] = segment_stats['repeat_users'] / segment_stats['total_users']

# Выведем статистику
print(segment_stats)

# Визуализируем долю пользователей, совершивших два и более заказа в зависимости от сегмента
plt.figure(figsize=(10, 6))
sns.barplot(x=segment_stats.index, y=segment_stats['repeat_rate'], palette='coolwarm')
plt.title('Доля пользователей с 2 и более заказами по сегментам по среднему количеству билетов')
plt.xlabel('Среднее количество билетов в заказе')
plt.ylabel('Доля пользователей с 2 и более заказами')
plt.show()

***Ответ:***
- Явно выражены пользователи, купившие в среднем 2-3 билета и 3-5 билетов. Процент пользователей, покупающих в среднем 5 и более билетов, составляет всего 3%.
- Также, стоит отметить, что вероятность повторных покупок пользователей в среднем покупающих 5 и более билетов составляет 18%. В других категориях доля возврата >50%.




<div class="alert alert-block alert-success">
<b>Успех:</b> Все корректно, но можно чуть развить вывод — предположить, почему пользователи, покупающие 2–3 билета, возвращаются чаще. Например, это могут быть небольшие компании или семьи, которые чаще ходят на мероприятия вместе, а значит, лояльность у них выше. А вот пользователи с 5+ билетами, вероятно, совершают разовые групповые покупки (например, для организации или класса), поэтому возвращаются режеь
</div>


---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
# Добавляем столбец с днем недели (0 - понедельник, 6 - воскресенье)
user_profile['first_order_weekday'] = user_profile['first_order_date'].dt.dayofweek

# Считаем статистику: количество пользователей и долю тех, кто совершил повторные заказы
weekday_stats = user_profile.groupby('first_order_weekday').agg(
    total_users=('user_id', 'count'),
    repeat_users=('is_two', 'sum')  # Пользователи с 2 и более заказами
)

# Рассчитываем долю пользователей с 2 и более заказами
weekday_stats['repeat_rate'] = weekday_stats['repeat_users'] / weekday_stats['total_users']

# Статистика по дням недели
print(weekday_stats)

# Визуализируем результат
plt.figure(figsize=(10, 6))
sns.barplot(x=weekday_stats.index, y=weekday_stats['repeat_rate'], palette='coolwarm')
plt.title('Доля пользователей с 2 и более заказами по дням недели')
plt.xlabel('День недели')
plt.ylabel('Доля пользователей с 2 и более заказами')
plt.xticks(ticks=range(7), labels=['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])
plt.show()

***Исходя из рассчетов и наглядной визуализации, можем смело опровергнуть, что дата покупки никак не влияет на вероятность возврата пользователя***

<div class="alert alert-block alert-success">
<b>Успех:</b> Верно, когда планируют досуг, но возвращаемость остаётся примерно одинаковой — это говорит о том, что день недели первой покупки не влияет на лояльность, а повторное использование сервиса определяется скорее качеством опыта и интересом к мероприятиям
</div>



---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
# Рассчитаем средний интервал между заказами для пользователей с 2-4 заказами и 5 и более заказами

# Фильтруем пользователей с 2-4 заказами
two_to_four_orders = user_profile[(user_profile['is_two'] == 1) & (user_profile['is_five'] == 0)]

# Фильтруем пользователей с 5 и более заказами
five_or_more_orders = user_profile[(user_profile['is_two'] == 1) & (user_profile['is_five'] == 1)]

# Рассчитываем средний интервал между заказами для каждой группы
# Для этого можно воспользоваться методом .diff() для нахождения разницы между заказами, затем .mean() для среднего интервала.

# Для группы 2-4 заказа
two_to_four_orders['days_between'] = two_to_four_orders['avg_days_between_orders']
avg_interval_two_to_four = two_to_four_orders['days_between'].mean()

# Для группы 5 и более заказов
five_or_more_orders['days_between'] = five_or_more_orders['avg_days_between_orders']
avg_interval_five_or_more = five_or_more_orders['days_between'].mean()

# Выведем средний интервал для обеих групп
print(f"Средний интервал между заказами (2–4 заказа): {avg_interval_two_to_four:.2f} дней")
print(f"Средний интервал между заказами (5 и более заказов): {avg_interval_five_or_more:.2f} дней")

# Визуализируем средний интервал между заказами для каждой группы
plt.figure(figsize=(8, 6))
sns.barplot(x=['2–4 заказа', '5 и более заказов'], y=[avg_interval_two_to_four, avg_interval_five_or_more], palette='Blues')
plt.title('Средний интервал между заказами для двух групп пользователей')
plt.xlabel('Группа пользователей')
plt.ylabel('Средний интервал (дни)')
plt.show()

- Частота повторных заказов зависит от времени между заказами: Чем меньше интервал между заказами, тем больше вероятность повторных заказов. Пользователи, совершившие 5 и более заказов, возвращаются гораздо быстрее и чаще, что делает их более лояльными.


- Сегмент с 2–4 заказами делает заказы реже, и это может быть связано с тем, что они менее лояльны или они находятся в стадии поиска лучшего предложения. Это открывает возможности для улучшения работы с этим сегментом, возможно, с помощью маркетинговых активностей или улучшения клиентского опыта.






<div class="alert alert-block alert-info">
<b>Совет:</b>  Важно иметь в виду, что среднее значение довольно сильно зависит от характера распределения, если есть какие-то сильные выбросы, они могут утянуть среднее значение вверх, хотя основаня масса значений будет гораздо ниже. Поэтому здесь важно было бы построить в том числе гистограммы, наложить их друг на друга (как мы это делали при сравнении выручки), чтобы видеть всю картину в данных.
</div>

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
!pip install phik

In [ ]:
from phik import phik_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Рассчитываем корреляцию phi_k между признаками профиля пользователя и total_orders
phik_corr = user_profile.phik_matrix()


<div class="alert alert-block alert-info">
<b>Совет:</b> В interval_cols желательно передавать интервальные колонки.
</div>


In [ ]:
# Визуализируем результат с помощью тепловой карты
plt.figure(figsize=(12, 8))
sns.heatmap(phik_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title('Корреляция признаков с total_orders (phi_k)')
plt.show()

# Выводим корреляцию для total_orders
print("Корреляция для total_orders:")
print(phik_corr['total_orders'])


<div class="alert alert-block alert-info">
<b>Совет:</b> Phik показывает от 0 до 1.  Расцветку для тепловой карты имеет смысл выбирать трехцветную, если диапазон возможных значений от -1 до 1 (например, синий, белый, красный, белый при этом в нуле), или двухцветную, если от 0 до 1 (в 0 - белый, в 1 - красный). Поскольку на 0 будет нейтральный цвет, величину корреляции можно будет воспринимать через интенсивность цвета, а по самому цвету - положительная она или отрицательная.
</div>


In [ ]:
def categorize_total_orders_count(row):
    if row['total_orders'] <= 1:
        return '1 заказ'
    elif 2 <= row['total_orders'] <= 4:
        return '2-4 заказа'
    else:
        return '5 и более заказов'

In [ ]:
user_profile['total_orders_segment'] = user_profile.apply(categorize_total_orders_count, axis=1)
user_profile

In [ ]:
phik_corr_segment = user_profile.phik_matrix()
print(phik_corr_segment)

In [ ]:
# Визуализируем результат с помощью тепловой карты
plt.figure(figsize=(12, 8))
sns.heatmap(phik_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title('Корреляция признаков с total_orders (phi_k)')
plt.show()

# Выводим корреляцию для total_orders
print("Корреляция для total_orders:")
print(phik_corr_segment['total_orders_segment'])

***Признаки, наиболее связанные с количеством заказов:***
- total_orders и total_orders_segment имеют корреляцию 1, что естественно, так как это одинаковые признаки, где total_orders_segment просто разделяет пользователей по количеству заказов.
- first_order_date и last_order_date имеют корреляцию 0.524768 и 0.559802 с total_orders соответственно, что может свидетельствовать о том, что более поздняя дата первого или последнего заказа может быть связана с большим числом заказов.
- avg_revenue (с корреляцией 0.298767) и avg_tickets_count (с корреляцией 0.358594) также положительно коррелируют с total_orders, что может означать, что пользователи, покупающие более дорогие билеты или покупающие больше билетов, делают больше заказов.
- avg_days_between_orders (с корреляцией 0.377128) также положительно коррелирует с total_orders, что говорит о том, что пользователи, которые делают заказы с более длительным интервалом, склонны совершать больше заказов в целом.




<div class="alert alert-block alert-success">
<b>Успех:</b> В выводах правильно сделан  явный акцент на том, что поведение клиентов во времени и количество билетов имеют наибольшую важность для повторных покупок
</div>


### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

# Общий вывод и рекомендации

#### 1. **Информация о данных и подготовка:**

В ходе анализа использовалась выгрузка данных с информацией о пользователях и их заказах на платформе Яндекс Афиша. После получения данных был проведен ряд этапов предобработки:

* **Фильтрация выбросов:** мы отфильтровали аномально высокие значения выручки с заказов, используя 99-й перцентиль для поля `revenue_rub`.
* **Конвертация валюты:** данные о выручке, представленные в казахстанских тенге, были приведены к российскому рублю с использованием актуального курса.
* **Обработка пропусков:** для столбца `days_since_prev` были пропущенные значения, что также было учтено при анализе.
* **Понижение размерности:** типы данных в числовых столбцах были оптимизированы для экономии памяти (например, `order_id` и `event_id` были приведены к типу `int32`, а выручка — к `float32`).

#### 2. **Основные результаты анализа:**

* **Общее количество пользователей:** В выборке представлено 20761 уникальных пользователей.
* **Распределение пользователей по числу заказов:** Большинство пользователей вернулись после совершения первого заказа, но также есть значительная доля пользователей совершивших 1 заказ . Это свидетельствует о разных уровнях вовлеченности пользователей.
* **Средняя выручка и количество билетов:** Средняя выручка и количество билетов показывают явную корреляцию с количеством заказов. Пользователи с более высокими значениями этих показателей, как правило, делают больше заказов.
* **Выбросы:** Наиболее значимые выбросы были в столбцах `revenue_rub` и `tickets_count`. Эти значения были отфильтрованы для обеспечения корректности анализа.

#### 3. **Признаки, влияющие на возврат пользователей:**

* **Тип мероприятия:** Возврат пользователей несильно зависит от типа первого мероприятия. Первой покупкой пользователей почти в половине случаев является билет на концерт.
* **Тип устройства:** Пользователи, чаще всего совершают первую покупку через телефон. Но на вероятность возрата пользователя это не влияет.
* **Регион:** Регионы с более высокой активностью пользователей, такие как Каменевский регион, имеют более высокую вероятность повторных покупок.

#### 4. **Влияние временных характеристик:**

* **День недели:** День недели первой покупки не оказывает влияние на вероятность повторного заказа.
* **Интервал между покупками:** Средний интервал между заказами был значительно меньше у пользователей с 5 и более заказами (11.13 дней), что подтверждает идею о том, что пользователи с высокой вовлеченностью делают заказы с меньшими интервалами между ними.

#### 5. **Корреляция признаков с количеством заказов:**

Признаки, такие как **средняя выручка (avg_revenue)** и **среднее количество билетов (avg_tickets_count)**, показывают положительную корреляцию с количеством заказов, что подтверждает гипотезу о том, что более крупные и дорогие заказы способствуют большему числу заказов. В то время как **тип устройства** и **тип мероприятия** имеют слабую корреляцию.

#### 6. **Рекомендации:**

1. **Целевая аудитория:** Следует сосредоточиться на пользователях, которые совершили 2 и более заказа. Это сегмент с высокой вероятностью повторных покупок, и для него можно запускать кампании по удержанию.
2. **Типы мероприятий:** Спортивные мероприятия и концерты привлекают более лояльных пользователей, что стоит учитывать при продвижении.
3. **Региональная активность:** Регионы с высокой активностью (например, Каменевский регион) следует выделить для персонализированных маркетинговых стратегий.
4. **Интервалы между покупками:** Предложить пользователям с длительными интервалами между покупками специальные предложения, чтобы уменьшить время между заказами.

Сфокусировавшись на этих сегментах, можно значительно улучшить удержание пользователей и повысить их вовлеченность на платформе.




<div class="alert alert-block alert-success">
<b>Успех:</b> Всегда приятно наблюдать подробный и структурированный итоговый вывод в конце работы!
</div>


### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

In [ ]:
!pip freeze > requirements.txt

https://github.com/doseevalex-bit/2_itog_project

<div class="alert alert-block alert-danger">
<b>Ошибка:</b> 
    
1) сама работа отсутвует на гитхабе
    
2) нельзя заливать .env - там же важные данные, которые мы обязаны скрыть (и в jupyter ноутбуке чувствительных данных быть не должно)

</div>

<div class="alert alert-block alert-warning">
<b>Изменения:</b> Залил проект на гитхаб. Но как убрать чувств. данные из юпитера если честно я не знаю) Этого вроде в теории не было. Если расскажешь - буду благодарен!
</div>



 <div class="alert alert-block alert-danger">
    
<b>Ответ[2]:</b>  Ключевой файл для - это .env. Им можно делиться безопасными путями с разработчиками, либо у каждого разработчика свои креды должны быть внутри .env.
    
Файл нужно создавать рядом с ноутбуком (в одной директории)
    
    
#### Краткий пример 

**1. Файл `.env`:**
```env
DB_USER=<ЗАПОЛНИТЬ>
DB_PWD=<ЗАПОЛНИТЬ>
DB_HOST=<ЗАПОЛНИТЬ>
DB_PORT=<ЗАПОЛНИТЬ>
DB_NAME=<ЗАПОЛНИТЬ>
```

**2. В Jupyter:**

в отдельной ячейке
```python
# Установка и импорт 
!pip install python-dotenv 
```
Затем

```python
from dotenv import load_dotenv
import os

# Загрузка .env
load_dotenv()

# Конфиг
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'db': os.getenv('DB_NAME')
}
```


    

</div>
